# 🔄 Topic 09: Cloud Pipeline Orchestration & PySpark Unit Testing

## 1. Orchestration Tools: Airflow (AWS MWAA) & Step Functions
Orchestrators trigger PySpark jobs on schedules or events, handle retries, manage task dependencies, and send alerts.

---

## 2. Hands-on: Unit Testing PySpark DataFrames with PyTest


In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Sample PySpark ETL Transformation Logic to Test
def transform_sales_data(df):
    return df.filter(F.col("amount") > 0) \
             .withColumn("revenue_taxed", F.round(F.col("amount") * 1.10, 2))

# --- UNIT TEST ---
def test_transform_sales_data():
    spark = SparkSession.builder.master("local[*]").appName("Testing_Demo").getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")
    
    # Input test DataFrame
    input_data = [("Tx1", 100.0), ("Tx2", -50.0), ("Tx3", 200.0)]
    df_input = spark.createDataFrame(input_data, ["id", "amount"])
    
    # Run transformation logic
    df_output = transform_sales_data(df_input)
    results = df_output.collect()
    
    # Assertions
    assert len(results) == 2, f"❌ Failed: Expected 2 rows, got {len(results)}"
    assert results[0]["revenue_taxed"] == 110.0, f"❌ Failed: Incorrect tax calculation!"
    print("✅ UNIT TEST PASSED: PySpark DataFrame transformation validated successfully!")

# Run test
test_transform_sales_data()


---

## 3. Production Apache Airflow DAG Blueprint (`big_data_pipeline_dag.py`)

```python
from airflow import DAG
from airflow.providers.amazon.aws.operators.emr import EmrAddStepsOperator
from datetime import datetime, timedelta

default_args = {
    'owner': 'data_engineering',
    'depends_on_past': False,
    'start_date': datetime(2026, 1, 1),
    'retries': 2,
    'retry_delay': timedelta(minutes=5),
}

with DAG(
    'cloud_big_data_spark_pipeline',
    default_args=default_args,
    schedule_interval='0 2 * * *', # Daily at 2 AM
    catchup=False
) as dag:

    run_spark_etl = EmrAddStepsOperator(
        task_id='run_pyspark_job',
        job_flow_id='j-EMRCLUSTER123',
        aws_conn_id='aws_default',
        steps=[{
            'Name': 'Execute PySpark ETL',
            'ActionOnFailure': 'CONTINUE',
            'HadoopJarStep': {
                'Jar': 'command-runner.jar',
                'Args': ['spark-submit', 's3://my-code-bucket/spark_etl_script.py'],
            },
        }]
    )
```
